In [1]:
import numpy as np
print(np.__version__)


import torch
print(torch.__version__)

import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

from tqdm.notebook import trange

import random
import math
import time

2.0.2
2.10.0+cu128


In [2]:
SIZE = 5
EMPTY = 0
P1 = 1
P2 = -1
max_moves = 200


class Quixo:
    def __init__(self):
        self.size = SIZE

        # Precompute perimeter positions
        self.perimeter = [
            (r, c)
            for r in range(SIZE)
            for c in range(SIZE)
            if r == 0 or r == SIZE - 1 or c == 0 or c == SIZE - 1
        ]

        # Max 3 moves per position → upper bound
        self.max_moves = len(self.perimeter) * 3
        self.action_map = []  # (r, c, dr, dc)
        self._build_action_map()

        self.action_size = len(self.action_map)

    def __repr__(self):
        return "Quixo"

    # ---------- State ----------

    def get_initial_state(self):
        return np.zeros((self.size, self.size), dtype=np.int8)

    def change_perspective(self, state, player):
        return state * player

    def get_opponent(self, player):
        return -player
    
    def get_opponent_value(self, value):
        return -value

    # ---------- Encoding ----------

    def get_encoded_state(self, state):
        encoded = np.stack(
            (state == P2, state == EMPTY, state == P1)
        ).astype(np.float32)

        if state.ndim == 3:
            encoded = np.swapaxes(encoded, 0, 1)

        return encoded

    # ---------- Action Space ----------

    def _build_action_map(self):
        """Enumerate all *legal* (position, direction) pairs."""
        for (r, c) in self.perimeter:

            # possible insertions excluding same-side
            candidates = []

            if c != 0:
                candidates.append((r, c, r, 0))
            if c != self.size - 1:
                candidates.append((r, c, r, self.size - 1))
            if r != 0:
                candidates.append((r, c, 0, c))
            if r != self.size - 1:
                candidates.append((r, c, self.size - 1, c))

            self.action_map.extend(candidates)

    def encode_action(self, r, c, dr, dc):
        return self.action_map.index((r, c, dr, dc))

    def decode_action(self, action):
        return self.action_map[action]

    # ---------- Legal Moves ----------

    def get_valid_moves(self, state):
        """
        Assumes state is already in current player's perspective (player = +1).
        """
        valid = np.zeros(self.action_size, dtype=np.uint8)

        for i, (r, c, dr, dc) in enumerate(self.action_map):
            piece = state[r, c]

            # can take EMPTY or own piece only
            if piece == EMPTY or piece == P1:
                valid[i] = 1

        return valid

    # ---------- Game Mechanics ----------

    def push(self, state, move):
        """
        Apply move assuming current player is +1 (canonical form).
        """
        r, c, dr, dc = move
        new = state.copy()

        # remove cube
        if r == dr:  # row move
            row = list(new[r, :])
            row.pop(c)

            # insert with forced overwrite
            if dc == 0:
                row.insert(0, P1)
            else:
                row.append(P1)

            new[r, :] = row

        else:  # column move
            col = list(new[:, c])
            col.pop(r)

            if dr == 0:
                col.insert(0, P1)
            else:
                col.append(P1)

            new[:, c] = col

        return new

    def get_next_state(self, state, action, player):
        """
        Applies action in *absolute* space, not canonical.
        """
        move = self.decode_action(action)

        # convert to canonical
        canonical = self.change_perspective(state, player)

        next_state = self.push(canonical, move)

        # revert perspective
        return self.change_perspective(next_state, player)

    # ---------- Win Detection ----------

    def has_line(self, state, player):
        # rows
        for r in range(self.size):
            if np.all(state[r, :] == player):
                return True

        # columns
        for c in range(self.size):
            if np.all(state[:, c] == player):
                return True

        # diagonals
        if np.all(np.diag(state) == player):
            return True

        if np.all(np.diag(np.fliplr(state)) == player):
            return True

        return False

    # ---------- Value ----------

    def get_value_and_terminated(self, state, player, move_count=None):
        """
        Evaluate from current player's perspective.

        If move_count is provided and >= max_moves with no winner, declare draw.
        """

        player_win = self.has_line(state, player)
        opp_win = self.has_line(state, -player)

        # critical rule: opponent line dominates
        if opp_win:
            return -1, True

        if player_win:
            return 1, True

        if move_count is not None and move_count >= max_moves:
            return 0, True

        return 0, False

In [3]:
# check number of valid moves for the initital state
game = Quixo()
initial_state = game.get_initial_state()
valid_moves = game.get_valid_moves(initial_state)
print(f"Number of valid moves from the initial state: {valid_moves.sum()}")

Number of valid moves from the initial state: 44


In [4]:
def print_board(state):
    symbols = {0: ".", 1: "X", -1: "O"}
    print()
    for r in range(state.shape[0]):
        print(" ".join(symbols[x] for x in state[r]))
    print()


def get_human_move(game, state):
    """
    Input format:
        r c side

    side ∈ {L, R, T, B}
    meaning:
        L → push from left
        R → push from right
        T → push from top
        B → push from bottom
    """

    side_map = {
        "L": lambda r, c: (r, c, r, 0),
        "R": lambda r, c: (r, c, r, game.size - 1),
        "T": lambda r, c: (r, c, 0, c),
        "B": lambda r, c: (r, c, game.size - 1, c),
    }

    valid = game.get_valid_moves(state)

    while True:
        try:
            raw = input("Enter move (r c side[L/R/T/B]): ").strip().split()
            r, c = int(raw[0]), int(raw[1])
            side = raw[2].upper()

            if side not in side_map:
                raise ValueError

            move = side_map[side](r, c)

            if move not in game.action_map:
                print("Illegal geometry (not a valid push direction).")
                continue

            action = game.encode_action(*move)

            if valid[action] == 0:
                print("Illegal move (violates rules).")
                continue

            return action

        except Exception:
            print("Invalid input. Format: r c side")


def play_game():
    game = Quixo()
    state = game.get_initial_state()
    player = 1  # X starts

    while True:
        print_board(state)

        print(f"Player {'X' if player == 1 else 'O'}")

        # canonical for move generation
        canonical = game.change_perspective(state, player)

        action = get_human_move(game, canonical)

        state = game.get_next_state(state, action, player)

        value, done = game.get_value_and_terminated(state, player)

        if done:
            print_board(state)

            if value == 1:
                print(f"Player {'X' if player == 1 else 'O'} wins")
            else:
                print(f"Player {'X' if player == 1 else 'O'} loses")

            break

        player = -player


# play_game()

In [5]:
class ResNet(nn.Module):
    def __init__(self, game, num_resBlocks, num_hidden, device):
        super().__init__()
        
        self.device = device
        self.startBlock = nn.Sequential(
            nn.Conv2d(3, num_hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )
        
        self.backBone = nn.ModuleList(
            [ResBlock(num_hidden) for i in range(num_resBlocks)]
        )
        
        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * game.size * game.size, game.action_size)
        )
        
        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 3, kernel_size=3, padding=1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * game.size * game.size, 1),
            nn.Tanh()
        )
        
        self.to(device)
        
    def forward(self, x):
        x = self.startBlock(x)
        for resBlock in self.backBone:
            x = resBlock(x)
        policy = self.policyHead(x)
        value = self.valueHead(x)
        return policy, value
        
        
class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)
        
    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        x = F.relu(x)
        return x
        

In [6]:
class Node:
    def __init__(self, game, args, state, parent=None, action_taken=None, prior=0, visit_count=0, move_count=0):
        self.game = game
        self.args = args
        self.state = state
        self.parent = parent
        self.action_taken = action_taken
        self.prior = prior
        self.move_count = move_count
        
        self.children = []
        
        self.visit_count = visit_count
        self.value_sum = 0
        
    def is_fully_expanded(self):
        return len(self.children) > 0
    
    def select(self):
        best_child = None
        best_ucb = -np.inf
        
        for child in self.children:
            ucb = self.get_ucb(child)
            if ucb > best_ucb:
                best_child = child
                best_ucb = ucb
                
        return best_child
    
    def get_ucb(self, child):
        if child.visit_count == 0:
            q_value = 0
        else:
            q_value = 1 - ((child.value_sum / child.visit_count) + 1) / 2
        return q_value + self.args['C'] * (math.sqrt(self.visit_count) / (child.visit_count + 1)) * child.prior
    
    def expand(self, policy):
        for action, prob in enumerate(policy):
            if prob > 0:
                child_state = self.state.copy()
                child_state = self.game.get_next_state(child_state, action, 1)
                child_state = self.game.change_perspective(child_state, player=-1)

                child = Node(
                    self.game,
                    self.args,
                    child_state,
                    self,
                    action,
                    prob,
                    move_count=self.move_count + 1,
                )
                self.children.append(child)
                
        return child
            
    def backpropagate(self, value):
        self.value_sum += value
        self.visit_count += 1
        
        value = self.game.get_opponent_value(value)
        if self.parent is not None:
            self.parent.backpropagate(value)  


class MCTS:
    def __init__(self, game, args, model):
        self.game = game
        self.args = args
        self.model = model
        
    @torch.no_grad()
    def search(self, state):
        root = Node(self.game, self.args, state, visit_count=1, move_count=0)
        
        policy, _ = self.model(
            torch.tensor(self.game.get_encoded_state(state), device=self.model.device).unsqueeze(0)
        )
        policy = torch.softmax(policy, axis=1).squeeze(0).cpu().numpy()
        policy = (1 - self.args['dirichlet_epsilon']) * policy + self.args['dirichlet_epsilon'] \
            * np.random.dirichlet([self.args['dirichlet_alpha']] * self.game.action_size)
        
        valid_moves = self.game.get_valid_moves(state)
        policy *= valid_moves
        policy /= np.sum(policy)
        root.expand(policy)
        
        for search in range(self.args['num_searches']):
            node = root
            
            while node.is_fully_expanded():
                node = node.select()
                
            value, is_terminal = self.game.get_value_and_terminated(node.state, 1, move_count=node.move_count)
            value = self.game.get_opponent_value(value)
            
            if not is_terminal:
                policy, value = self.model(
                    torch.tensor(self.game.get_encoded_state(node.state), device=self.model.device).unsqueeze(0)
                )
                policy = torch.softmax(policy, axis=1).squeeze(0).cpu().numpy()
                valid_moves = self.game.get_valid_moves(node.state)
                policy *= valid_moves
                policy /= np.sum(policy)
                
                value = value.item()
                
                node.expand(policy)
                
            node.backpropagate(value)    
            
            
        action_probs = np.zeros(self.game.action_size)
        for child in root.children:
            action_probs[child.action_taken] = child.visit_count
        action_probs /= np.sum(action_probs)
        return action_probs
        

In [7]:
class AlphaZero:
    def __init__(self, model, optimizer, game, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game
        self.args = args
        self.mcts = MCTS(game, args, model)
        
    def selfPlay(self):
        memory = []
        player = 1
        state = self.game.get_initial_state()
        move_count = 0
        
        while True:
            neutral_state = self.game.change_perspective(state, player)
            action_probs = self.mcts.search(neutral_state)
            
            memory.append((neutral_state, action_probs, player))
            
            eps = 1e-8
            temperature_action_probs = action_probs ** (1 / self.args['temperature'])
            temperature_action_probs = temperature_action_probs / (temperature_action_probs.sum() + eps)            
            action = np.random.choice(self.game.action_size, p=temperature_action_probs)
						
            state = self.game.get_next_state(state, action, player)
            move_count += 1
            
            value, is_terminal = self.game.get_value_and_terminated(state, player, move_count=move_count)
            
            if is_terminal:
                returnMemory = []
                for hist_neutral_state, hist_action_probs, hist_player in memory:
                    hist_outcome = value if hist_player == player else self.game.get_opponent_value(value)
                    returnMemory.append((
                        self.game.get_encoded_state(hist_neutral_state),
                        hist_action_probs,
                        hist_outcome
                    ))
                return returnMemory
            
            player = self.game.get_opponent(player)
                
    def train(self, memory):
        random.shuffle(memory)
        for batchIdx in range(0, len(memory), self.args['batch_size']):
            sample = memory[batchIdx:min(len(memory) - 1, batchIdx + self.args['batch_size'])] # Change to memory[batchIdx:batchIdx+self.args['batch_size']] in case of an error
            state, policy_targets, value_targets = zip(*sample)
            
            state, policy_targets, value_targets = np.array(state), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)
            
            state = torch.tensor(state, dtype=torch.float32, device=self.model.device)
            policy_targets = torch.tensor(policy_targets, dtype=torch.float32, device=self.model.device)
            value_targets = torch.tensor(value_targets, dtype=torch.float32, device=self.model.device)
            
            out_policy, out_value = self.model(state)
            
            policy_loss = F.cross_entropy(out_policy, policy_targets)
            value_loss = F.mse_loss(out_value, value_targets)
            loss = policy_loss + value_loss
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
    
    def learn(self):
        for iteration in range(self.args['num_iterations']):
            memory = []
            
            self.model.eval()
            for selfPlay_iteration in trange(self.args['num_selfPlay_iterations']):
                memory += self.selfPlay()
                
            self.model.train()
            for epoch in trange(self.args['num_epochs']):
                self.train(memory)
            
            torch.save(self.model.state_dict(), f"model_{iteration}_{self.game}.pt")
            torch.save(self.optimizer.state_dict(), f"optimizer_{iteration}_{self.game}.pt")

In [8]:
class MCTSParallel:
    def __init__(self, game, args, model):
        self.game = game
        self.args = args
        self.model = model
        
    @torch.no_grad()
    def search(self, states, spGames):
        policy, _ = self.model(
            torch.tensor(self.game.get_encoded_state(states), device=self.model.device)
        )
        policy = torch.softmax(policy, axis=1).cpu().numpy()
        policy = (1 - self.args['dirichlet_epsilon']) * policy + self.args['dirichlet_epsilon'] \
            * np.random.dirichlet([self.args['dirichlet_alpha']] * self.game.action_size, size=policy.shape[0])
        
        for i, spg in enumerate(spGames):
            spg_policy = policy[i]
            valid_moves = self.game.get_valid_moves(states[i])
            spg_policy *= valid_moves
            spg_policy /= np.sum(spg_policy)

            spg.root = Node(self.game, self.args, states[i], visit_count=1, move_count=spg.move_count)
            spg.root.expand(spg_policy)
        
        for search in range(self.args['num_searches']):
            for spg in spGames:
                spg.node = None
                node = spg.root

                while node.is_fully_expanded():
                    node = node.select()

                value, is_terminal = self.game.get_value_and_terminated(node.state, 1, move_count=node.move_count)
                value = self.game.get_opponent_value(value)
                
                if is_terminal:
                    node.backpropagate(value)
                    
                else:
                    spg.node = node
                    
            expandable_spGames = [mappingIdx for mappingIdx in range(len(spGames)) if spGames[mappingIdx].node is not None]
                    
            if len(expandable_spGames) > 0:
                states = np.stack([spGames[mappingIdx].node.state for mappingIdx in expandable_spGames])
                
                policy, value = self.model(
                    torch.tensor(self.game.get_encoded_state(states), device=self.model.device)
                )
                policy = torch.softmax(policy, axis=1).cpu().numpy()
                value = value.cpu().numpy()
                
            for i, mappingIdx in enumerate(expandable_spGames):
                node = spGames[mappingIdx].node
                spg_policy, spg_value = policy[i], value[i]
                
                valid_moves = self.game.get_valid_moves(node.state)
                spg_policy *= valid_moves
                spg_policy /= np.sum(spg_policy)

                node.expand(spg_policy)
                node.backpropagate(spg_value)

In [12]:
class AlphaZeroParallel:
    def __init__(self, model, optimizer, game, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game
        self.args = args
        self.mcts = MCTSParallel(game, args, model)
        
    def selfPlay(self):
        return_memory = []
        player = 1
        spGames = [SPG(self.game) for spg in range(self.args['num_parallel_games'])]
        
        while len(spGames) > 0:
            states = np.stack([spg.state for spg in spGames])
            neutral_states = self.game.change_perspective(states, player)
            
            self.mcts.search(neutral_states, spGames)
            
            for i in range(len(spGames))[::-1]:
                spg = spGames[i]
                
                action_probs = np.zeros(self.game.action_size)
                for child in spg.root.children:
                    action_probs[child.action_taken] = child.visit_count
                action_probs /= np.sum(action_probs)

                spg.memory.append((spg.root.state, action_probs, player))

                eps = 1e-8
                temperature_action_probs = action_probs ** (1 / self.args['temperature'])
                temperature_action_probs = temperature_action_probs / (temperature_action_probs.sum() + eps)
                
                action = np.random.choice(self.game.action_size, p=temperature_action_probs) # Divide temperature_action_probs with its sum in case of an error

                spg.state = self.game.get_next_state(spg.state, action, player)
                spg.move_count += 1

                value, is_terminal = self.game.get_value_and_terminated(spg.state, player, move_count=spg.move_count)

                if is_terminal:
                    for hist_neutral_state, hist_action_probs, hist_player in spg.memory:
                        hist_outcome = value if hist_player == player else self.game.get_opponent_value(value)
                        return_memory.append((
                            self.game.get_encoded_state(hist_neutral_state),
                            hist_action_probs,
                            hist_outcome
                        ))
                    del spGames[i]
                    
            player = self.game.get_opponent(player)
            
        return return_memory
                
    def train(self, memory):
        random.shuffle(memory)
        for batchIdx in range(0, len(memory), self.args['batch_size']):
            sample = memory[batchIdx:min(len(memory) - 1, batchIdx + self.args['batch_size'])] # Change to memory[batchIdx:batchIdx+self.args['batch_size']] in case of an error
            state, policy_targets, value_targets = zip(*sample)
            
            state, policy_targets, value_targets = np.array(state), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)
            
            state = torch.tensor(state, dtype=torch.float32, device=self.model.device)
            policy_targets = torch.tensor(policy_targets, dtype=torch.float32, device=self.model.device)
            value_targets = torch.tensor(value_targets, dtype=torch.float32, device=self.model.device)
            
            out_policy, out_value = self.model(state)
            
            policy_loss = F.cross_entropy(out_policy, policy_targets)
            value_loss = F.mse_loss(out_value, value_targets)
            loss = policy_loss + value_loss
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()


    def learn(self):
        for iteration in range(self.args['num_iterations']):
            memory = []
            
            self.model.eval()
            start_time = time.time()
            total_states = 0
            
            # We calculate how many loops of parallel games we need
            num_loops = self.args['num_selfPlay_iterations'] // self.args['num_parallel_games']
            
            print(f"--- Iteration {iteration} Self-Play ---")
            for selfPlay_iteration in trange(num_loops):
                print(f"Self-Play Loop {selfPlay_iteration + 1}/{num_loops}")
                batch_memory = self.selfPlay()
                memory += batch_memory
                total_states += len(batch_memory)
                
            end_time = time.time()
            elapsed = end_time - start_time
            gps = self.args['num_selfPlay_iterations'] / elapsed
            sps = total_states / elapsed
            
            print(f"Performance Metrics:")
            print(f" - Total Time: {elapsed:.2f}s")
            print(f" - Games per Second: {gps:.2f}")
            print(f" - States per Second: {sps:.2f}")
            
            self.model.train()
            print(f"--- Iteration {iteration} Training ---")
            for epoch in trange(self.args['num_epochs']):
                print(f"Epoch {epoch + 1}/{self.args['num_epochs']}")
                self.train(memory)
            
            torch.save(self.model.state_dict(), f"model_{iteration}_{self.game}.pt")
            torch.save(self.optimizer.state_dict(), f"optimizer_{iteration}_{self.game}.pt")
    # def learn(self):
    #     for iteration in range(self.args['num_iterations']):
    #         memory = []
            
    #         self.model.eval()
    #         for selfPlay_iteration in trange(self.args['num_selfPlay_iterations'] // self.args['num_parallel_games']):
    #             memory += self.selfPlay()
                
    #         self.model.train()
    #         for epoch in trange(self.args['num_epochs']):
    #             self.train(memory)
            
    #         torch.save(self.model.state_dict(), f"model_{iteration}_{self.game}.pt")
    #         torch.save(self.optimizer.state_dict(), f"optimizer_{iteration}_{self.game}.pt")
            
class SPG:
    def __init__(self, game):
        self.state = game.get_initial_state()
        self.memory = []
        self.root = None
        self.node = None
        self.move_count = 0

In [ ]:
game = Quixo()

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")

model = ResNet(game, 9, 128, device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)

args = {
    'C': 2,
    'num_searches': 200, # 600
    'num_iterations': 2, # 8
    'num_selfPlay_iterations': 512, # 500
    'num_parallel_games': 128, # 100
    'num_epochs': 4, # 4
    'batch_size': 128,
    'temperature': 1.25,
    'dirichlet_epsilon': 0.25,
    'dirichlet_alpha': 0.3
}

alphaZero = AlphaZeroParallel(model, optimizer, game, args)
alphaZero.learn()

Using device: cuda
--- Iteration 0 Self-Play ---


  0%|          | 0/4 [00:00<?, ?it/s]

Self-Play Loop 1/4


In [11]:
import numpy as np
import torch

def print_board(state):
    symbols = {0: ".", 1: "X", -1: "O"}
    print()
    for r in range(state.shape[0]):
        print(" ".join(symbols[x] for x in state[r]))
    print()


def get_human_action(game, state):
    side_map = {
        "L": lambda r, c: (r, c, r, 0),
        "R": lambda r, c: (r, c, r, game.size - 1),
        "T": lambda r, c: (r, c, 0, c),
        "B": lambda r, c: (r, c, game.size - 1, c),
    }

    valid = game.get_valid_moves(state)

    while True:
        try:
            raw = input("Enter move (r c side[L/R/T/B]): ").strip().split()
            r, c = int(raw[0]), int(raw[1])
            side = raw[2].upper()

            if side not in side_map:
                raise ValueError

            move = side_map[side](r, c)

            if move not in game.action_map:
                print("Invalid geometry")
                continue

            action = game.encode_action(*move)

            if valid[action] == 0:
                print("Illegal move")
                continue

            return action

        except Exception:
            print("Invalid input format")


# --- Setup ---
game = Quixo()
player = 1

args = {
    'C': 2,
    'num_searches': 600,
    'dirichlet_epsilon': 0.,
    'dirichlet_alpha': 0.3
}

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model = ResNet(game, 9, 128, device)
model.load_state_dict(torch.load("model_7_Quixo.pt", map_location=device))
model.eval()

mcts = MCTS(game, args, model)

state = game.get_initial_state()


# --- Game Loop ---
while True:
    print_board(state)

    if player == 1:
        print("Player X (human)")
        canonical = game.change_perspective(state, player)
        action = get_human_action(game, canonical)

    else:
        print("Player O (AI)")
        neutral_state = game.change_perspective(state, player)
        mcts_probs = mcts.search(neutral_state)
        action = np.argmax(mcts_probs)

    state = game.get_next_state(state, action, player)

    value, is_terminal = game.get_value_and_terminated(state, player)

    if is_terminal:
        print_board(state)

        if value == 1:
            print(f"Player {'X' if player == 1 else 'O'} wins")
        else:
            print(f"Player {'X' if player == 1 else 'O'} loses")

        break

    player = game.get_opponent(player)

FileNotFoundError: [Errno 2] No such file or directory: 'model_7_Quixo.pt'